# 02 - HITL Active Learning Loop

Re-run this notebook after each human labelling round.
Increment `PENDING_BATCH_TO_PROCESS` each time (1 → 2 → 3 → 4).

In [ ]:
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder        = BASE_PATH / 'Raw Data/Twits/'
test_folder         = BASE_PATH / 'Raw Data/'
datasets_folder     = BASE_PATH / 'Data Sets'
cleanedds_folder    = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder     = BASE_PATH / 'Data Sets/Networks/'
literature_folder   = BASE_PATH / 'Literature/'
topic_models_folder = BASE_PATH / 'Models/Topic Modeling/'
hitl_folder = datasets_folder / 'Classifiers_Data' / 'HITL'

In [ ]:
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'transformers', 'torch',
                    'sentence-transformers', 'lightgbm', 'scikit-learn', 'datasets'])
else:
    print('Running locally: skipping Colab setup.')

In [ ]:
import time
import glob
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import lightgbm as lgb
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, pipeline)
from datasets import Dataset

## Configuration

In [ ]:
PENDING_BATCH_TO_PROCESS = 1   # change to 2, 3, 4 for subsequent iterations
NEXT_BATCH_PATH = hitl_folder / f'hitl_pending_batch_{PENDING_BATCH_TO_PROCESS:02d}.pkl'

## 1. Load All Labeled Data

In [ ]:
labeled_files = sorted(glob.glob(str(hitl_folder / 'hitl_review_batch_*.csv')))
print(f'Found {len(labeled_files)} labeled batch(es).')

dfs = []
for f in labeled_files:
    tmp = pd.read_csv(f)
    if 'human_label' in tmp.columns:
        dfs.append(tmp.dropna(subset=['human_label']))

if not dfs:
    raise ValueError('No labeled data found. Label hitl_review_batch_00.csv first.')

train_df = pd.concat(dfs, ignore_index=True)
train_df['text'] = train_df['text'].astype(str)
print(f'Total labeled examples: {len(train_df):,}')

X_tr, X_val, y_tr, y_val = train_test_split(
    train_df['text'], train_df['human_label'], test_size=0.1, random_state=42)

## 2. Fast Models (TF-IDF + Logistic Regression & LightGBM)

In [ ]:
t0  = time.time()
vec = TfidfVectorizer(max_features=25_000)
Xtr_v  = vec.fit_transform(X_tr)
Xval_v = vec.transform(X_val)
print(f'TF-IDF vectorisation: {time.time()-t0:.1f}s')

t0 = time.time()
lr = LogisticRegression(max_iter=1000)
lr.fit(Xtr_v, y_tr)
lr_tr = time.time()-t0
t0 = time.time()
lr_acc = accuracy_score(y_val, lr.predict(Xval_v))
lr_inf = time.time()-t0
print(f'LR   train {lr_tr:.1f}s | infer {lr_inf:.2f}s | acc {lr_acc:.4f}')

t0 = time.time()
lgbm = lgb.LGBMClassifier(n_estimators=200, random_state=42)
lgbm.fit(Xtr_v, y_tr)
lg_tr = time.time()-t0
t0 = time.time()
lg_acc = accuracy_score(y_val, lgbm.predict(Xval_v))
lg_inf = time.time()-t0
print(f'LGBM train {lg_tr:.1f}s | infer {lg_inf:.2f}s | acc {lg_acc:.4f}')

## 3. Twitter-RoBERTa Fine-Tuning

`cardiffnlp/twitter-roberta-base` — pre-trained on 58 M tweets.

In [ ]:
model_name = 'cardiffnlp/twitter-roberta-base'
tokenizer  = AutoTokenizer.from_pretrained(model_name)

unique_labels = list(train_df['human_label'].unique())
label2id = {str(v): i for i, v in enumerate(unique_labels)}
id2label  = {i: str(v) for i, v in enumerate(unique_labels)}

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

hf_train = Dataset.from_dict({
    'text':  X_tr.tolist(),
    'label': [label2id[str(y)] for y in y_tr]}).map(tokenize, batched=True)
hf_val = Dataset.from_dict({
    'text':  X_val.tolist(),
    'label': [label2id[str(y)] for y in y_val]}).map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir='./roberta_results',
    evaluation_strategy='epoch', save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=3, weight_decay=0.01, load_best_model_at_end=True)

def compute_metrics(ep):
    logits, labels = ep
    return {'accuracy': accuracy_score(labels, np.argmax(logits, axis=-1))}

trainer = Trainer(model=model, args=args,
                  train_dataset=hf_train, eval_dataset=hf_val,
                  compute_metrics=compute_metrics)

In [ ]:
t0 = time.time()
trainer.train()
print(f'RoBERTa train time: {time.time()-t0:.1f}s')

res = trainer.evaluate()
print(f'RoBERTa val accuracy: {res["eval_accuracy"]:.4f}')

best_path = hitl_folder / 'best_roberta_model'
trainer.save_model(str(best_path))
tokenizer.save_pretrained(str(best_path))
print(f'Model saved to {best_path}')

## 4. Predict Next Batch and Export Active-Learning Sample

In [ ]:
assert NEXT_BATCH_PATH.exists(), f'Batch not found: {NEXT_BATCH_PATH}'

pending = pd.read_pickle(NEXT_BATCH_PATH)
print(f'Running inference on {len(pending):,} tweets...')

clf_pipe = pipeline('text-classification', model=trainer.model,
                    tokenizer=tokenizer,
                    device=0 if torch.cuda.is_available() else -1,
                    return_all_scores=True)

preds = []
t0 = time.time()
for i in range(0, len(pending), 500):
    preds.extend(clf_pipe(pending['text'].iloc[i:i+500].astype(str).tolist()))
print(f'Inference done in {time.time()-t0:.1f}s')

pending['predicted_label'] = [max(s, key=lambda x: x['score'])['label'] for s in preds]
pending['confidence']      = [max(s, key=lambda x: x['score'])['score'] for s in preds]

In [ ]:
N_UNCERTAIN = 5_000
N_RANDOM    = 5_000

uncertain = pending.nsmallest(min(N_UNCERTAIN, len(pending)), 'confidence')
pool      = pending.drop(uncertain.index)
random_s  = pool.sample(n=min(N_RANDOM, len(pool)), random_state=42)

export = pd.concat([uncertain, random_s]).sample(frac=1, random_state=42)
export['human_label'] = np.nan
export['text'] = export['text'].astype(str).str.replace('\n', ' ', regex=False)

out = hitl_folder / f'hitl_review_batch_{PENDING_BATCH_TO_PROCESS:02d}.csv'
export.to_csv(out, index=False)
print(f'Exported {len(export):,} tweets for review to {out}')
print('Next: fill human_label, save, increment PENDING_BATCH_TO_PROCESS, re-run.')